In [1]:
# =============================================================================
# CELL 1 — Setup and Imports
# =============================================================================
!pip install implicit huggingface_hub -q

import os
import json
import time
import pickle
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Optional, Literal
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from huggingface_hub import HfFileSystem
import implicit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings

# Try to get HF_TOKEN if running on Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_TOKEN = None

HF_REPO  = "vngclinh/goodreads-preprocessed"
hf_fs    = HfFileSystem(token=HF_TOKEN)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def hf_read(path, cols=None):
    with hf_fs.open(f"datasets/{HF_REPO}/{path}", "rb") as f:
        return pd.read_parquet(f, columns=cols)

print("Setup OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 102.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


Setup OK


In [2]:
# =============================================================================
# CELL 2 — ChainRec Data Loader Definition
# =============================================================================

@dataclass
class DataConfig:
    raw_csv: str = "/kaggle/working/goodreads_interactions.csv"
    out_dir: str = "/kaggle/working/processed"
    n_stages: int = 4
    recommend_threshold: int = 4
    min_user_inter: int = 5
    min_item_inter: int = 5
    require_final_stage: bool = True
    sample_n_users: Optional[int] = None
    sample_seed: int = 1234
    n_val_users: int = 5000
    n_test_users: int = 5000

class GoodreadsChainLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
        Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
        self.user_idx: dict = {}
        self.item_idx: dict = {}
        self.n_user: int = 0
        self.n_item: int = 0
        self.interactions: np.ndarray = None
        self.user_item_map: dict = defaultdict(set)

    def _first_pass_counts(self, chunksize=5_000_000):
        print("[1/4] First pass: counting interactions...")
        user_counts, item_counts = defaultdict(int), defaultdict(int)
        n_rows = 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                  usecols=["user_id","book_id","is_read","rating"],
                                  chunksize=chunksize,
                                  dtype={"user_id":np.int32,"book_id":np.int32,
                                         "is_read":np.int8,"rating":np.int8}):
            for u, c in chunk["user_id"].value_counts().items(): user_counts[u] += c
            for i, c in chunk["book_id"].value_counts().items(): item_counts[i] += c
            n_rows += len(chunk)
            print(f"    ...{n_rows:,} rows", end="\r")
        print(f"\n    total={n_rows:,}, users={len(user_counts):,}, items={len(item_counts):,}")
        return user_counts, item_counts

    def _build_id_maps(self, user_counts, item_counts):
        print("[2/4] Filtering and building id maps...")
        valid_users = {u for u,c in user_counts.items() if c >= self.cfg.min_user_inter}
        valid_items = {i for i,c in item_counts.items() if c >= self.cfg.min_item_inter}
        print(f"    valid users={len(valid_users):,}, items={len(valid_items):,}")
        if self.cfg.sample_n_users is not None:
            rng = np.random.default_rng(self.cfg.sample_seed)
            lst = sorted(valid_users)
            if len(lst) > self.cfg.sample_n_users:
                valid_users = set(rng.choice(lst, size=self.cfg.sample_n_users, replace=False).tolist())
                print(f"    sampled {len(valid_users):,} users")
        self.user_idx = {u:i for i,u in enumerate(sorted(valid_users))}
        self.item_idx = {b:i for i,b in enumerate(sorted(valid_items))}
        self.n_user = len(self.user_idx)
        self.n_item = len(self.item_idx)
        return valid_users, valid_items

    def _second_pass_extract(self, valid_users, valid_items, chunksize=5_000_000):
        print("[3/4] Second pass: extracting chains...")
        rows, n_rows = [], 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                  usecols=["user_id","book_id","is_read","rating"],
                                  chunksize=chunksize,
                                  dtype={"user_id":np.int32,"book_id":np.int32,
                                         "is_read":np.int8,"rating":np.int8}):
            mask = chunk["user_id"].isin(valid_users) & chunk["book_id"].isin(valid_items)
            sub = chunk[mask]
            if len(sub) == 0:
                n_rows += len(chunk)
                continue
            stage = np.zeros(len(sub), dtype=np.int8)
            stage[sub["is_read"].values == 1] = 1
            rated = sub["rating"].values > 0
            stage[rated] = np.maximum(stage[rated], 2)
            recom = sub["rating"].values >= self.cfg.recommend_threshold
            stage[recom] = np.maximum(stage[recom], 3)
            u_idx = sub["user_id"].map(self.user_idx).values
            i_idx = sub["book_id"].map(self.item_idx).values
            rows.append(np.stack([u_idx, i_idx, stage], axis=1))
            n_rows += len(chunk)
            print(f"    ...{n_rows:,} rows, kept {sum(len(r) for r in rows):,}", end="\r")
        self.interactions = np.concatenate(rows).astype(np.int32)
        print(f"\n    kept {len(self.interactions):,} interactions")

    def _post_filter_and_index(self):
        print("[4/4] Post-filter...")
        if self.cfg.require_final_stage:
            last = self.cfg.n_stages - 1
            users_with_last = np.unique(self.interactions[self.interactions[:,2]==last, 0])
            keep = np.isin(self.interactions[:,0], users_with_last)
            self.interactions = self.interactions[keep]
            kept = np.unique(self.interactions[:,0])
            remap = {old:new for new,old in enumerate(kept)}
            self.interactions[:,0] = np.array([remap[u] for u in self.interactions[:,0]], dtype=np.int32)
            inv = {v:k for k,v in self.user_idx.items()}
            self.user_idx = {inv[old]:new for old,new in remap.items()}
            self.n_user = len(kept)
            print(f"    {self.n_user:,} users, {len(self.interactions):,} interactions")
        for u,i,_ in self.interactions:
            self.user_item_map[int(u)].add(int(i))
        stage_counts = np.bincount(self.interactions[:,2], minlength=self.cfg.n_stages)
        print("    stage dist: " + ", ".join(f"stage{l}={c:,}" for l,c in enumerate(stage_counts)))

    def split_train_test(self):
        print("Splitting...")
        rng = np.random.default_rng(self.cfg.sample_seed)
        last = self.cfg.n_stages - 1
        user_final = defaultdict(list)
        for idx,(u,i,s) in enumerate(self.interactions):
            if s == last: user_final[int(u)].append(idx)
        eligible = [u for u,idxs in user_final.items() if len(idxs) >= 3]
        rng.shuffle(eligible)
        n_val  = min(self.cfg.n_val_users,  len(eligible) // 2)
        n_test = min(self.cfg.n_test_users, len(eligible) - n_val)
        val_idx  = [rng.choice(user_final[u]) for u in eligible[:n_val]]
        test_idx = [rng.choice(user_final[u]) for u in eligible[n_val:n_val+n_test]]
        holdout = set(val_idx) | set(test_idx)
        mask = np.ones(len(self.interactions), dtype=bool)
        mask[list(holdout)] = False
        self.data_train = self.interactions[mask]
        self.data_val   = self.interactions[val_idx]
        self.data_test  = self.interactions[test_idx]
        print(f"    train={len(self.data_train):,}, val={len(self.data_val):,}, test={len(self.data_test):,}")

    def build(self):
        uc, ic = self._first_pass_counts()
        vu, vi = self._build_id_maps(uc, ic)
        self._second_pass_extract(vu, vi)
        self._post_filter_and_index()
        self.split_train_test()
        return self

    def save(self):
        out = Path(self.cfg.out_dir)
        np.save(out/"data_train.npy", self.data_train)
        np.save(out/"data_val.npy",   self.data_val)
        np.save(out/"data_test.npy",  self.data_test)
        pickle.dump(self.user_idx,        open(out/"user_idx.pkl","wb"))
        pickle.dump(self.item_idx,        open(out/"item_idx.pkl","wb"))
        pickle.dump(dict(self.user_item_map), open(out/"user_item_map.pkl","wb"))
        json.dump({"n_user":self.n_user,"n_item":self.n_item,"n_stage":self.cfg.n_stages,
                   "config":asdict(self.cfg)},
                  open(out/"meta.json","w"), indent=2)
        print(f"Saved to {out}/")

In [3]:
# =============================================================================
# CELL 3 — Run ChainRec Data Loader (Load processed if exists)
# =============================================================================

PROC = Path("/kaggle/working/processed")

if not (PROC / "data_train.npy").exists():
    print("Downloading dataset...")
    if not os.path.exists("/kaggle/working/goodreads_interactions.csv"):
        os.system("wget -q https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv -O /kaggle/working/goodreads_interactions.csv")
        print("Download xong interactions!")
        
    cfg_data = DataConfig(
        raw_csv        = "/kaggle/working/goodreads_interactions.csv",
        out_dir        = "/kaggle/working/processed",
        sample_n_users = 50000,
        n_val_users    = 5000,
        n_test_users   = 5000,
    )
    loader = GoodreadsChainLoader(cfg_data).build()
    loader.save()

# Download mapping files for Hybrid matching
if not os.path.exists("/kaggle/working/user_id_map.csv"):
    os.system("wget -q https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/user_id_map.csv -O /kaggle/working/user_id_map.csv")
if not os.path.exists("/kaggle/working/book_id_map.csv"):
    os.system("wget -q https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/book_id_map.csv -O /kaggle/working/book_id_map.csv")

data_train = np.load(PROC / "data_train.npy")
data_val   = np.load(PROC / "data_val.npy")
data_test  = np.load(PROC / "data_test.npy")
with open(PROC / "user_item_map.pkl", "rb") as f:
    user_item_map = pickle.load(f)
with open(PROC / "user_idx.pkl", "rb") as f:
    chain_user_idx = pickle.load(f)
with open(PROC / "item_idx.pkl", "rb") as f:
    chain_item_idx = pickle.load(f)
meta = json.loads((PROC / "meta.json").read_text())

# Create inverse mappings: ChainRec Index -> Goodreads CSV Integer ID
inv_user_idx = {v: k for k, v in chain_user_idx.items()}
inv_item_idx = {v: k for k, v in chain_item_idx.items()}

# Create CSV Integer ID -> Original String ID mappings
print("Loading ID map files...")
user_id_map_df = pd.read_csv("/kaggle/working/user_id_map.csv")
book_id_map_df = pd.read_csv("/kaggle/working/book_id_map.csv")
csv_int_to_str_user = dict(zip(user_id_map_df.iloc[:,0], user_id_map_df.iloc[:,1].astype(str)))
csv_int_to_str_book = dict(zip(book_id_map_df.iloc[:,0], book_id_map_df.iloc[:,1].astype(str)))
del user_id_map_df, book_id_map_df

print(f"n_user={meta['n_user']:,}, n_item={meta['n_item']:,}, n_stage={meta['n_stage']}")
print(f"train={len(data_train):,}, val={len(data_val):,}, test={len(data_test):,}")

Download xong interactions!
[1/4] First pass: counting interactions...
    ...228,648,342 rows
    total=228,648,342, users=876,145, items=2,360,650
[2/4] Filtering and building id maps...
    valid users=812,035, items=1,567,258
    sampled 50,000 users
[3/4] Second pass: extracting chains...
    ...228,648,342 rows, kept 13,805,205
    kept 13,805,205 interactions
[4/4] Post-filter...
    48,307 users, 13,705,971 interactions
    stage dist: stage0=6,894,228, stage1=448,217, stage2=1,910,069, stage3=4,453,457
Splitting...
    train=13,695,971, val=5,000, test=5,000
Saved to /kaggle/working/processed/
Loading ID map files...
n_user=48,307, n_item=1,567,258, n_stage=4
train=13,695,971, val=5,000, test=5,000


In [4]:
# =============================================================================
# CELL 4 — ChainRec Model Definition
# =============================================================================

@dataclass
class ModelConfig:
    n_user: int
    n_item: int
    n_stage: int       = 4
    embed_dim: int     = 64
    beta: float        = 1.0
    learn_beta: bool   = True
    l2: float          = 0.01
    lr: float          = 0.001
    batch_size: int    = 1024
    n_neg: int         = 1
    n_epochs: int      = 50
    patience: int      = 5
    sampler: Literal["uniform", "stagewise"] = "uniform"
    device: str        = "cuda" if torch.cuda.is_available() else "cpu"

class ChainRecModel(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        log_beta_init  = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta:
            self.log_beta = nn.Parameter(log_beta_init)
        else:
            self.register_buffer("log_beta", log_beta_init)
        for emb in [self.user_emb, self.item_emb, self.stage_emb]:
            nn.init.xavier_uniform_(emb.weight)
        for bias in [self.b_user, self.b_item]:
            nn.init.zeros_(bias.weight)

    @property
    def beta(self):
        return torch.clamp(self.log_beta.exp(), min=1.0)

    def _intention_score(self, u, i, l):
        return (self.stage_emb(l) * self.item_emb(i) * self.user_emb(u)).sum(-1)

    def _rectified(self, delta):
        b = self.beta
        return F.softplus(b * delta) / b

    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rectified(self._intention_score(u, i, l_t))
        return bias + acc

In [5]:
# =============================================================================
# CELL 5 — Load or Initialize ChainRec Model
# =============================================================================

cfg = ModelConfig(
    n_user     = meta["n_user"],
    n_item     = meta["n_item"],
    n_stage    = meta["n_stage"],
)

chain_model = ChainRecModel(cfg).to(cfg.device)

# Load pretrained weights if available
model_path = "/kaggle/input/datasets/edsjkgdsk/chainrec-model/chainrec_best.pt"
if os.path.exists(model_path):
    print("Loading pretrained ChainRec model...")
    chain_model.load_state_dict(torch.load(model_path, map_location=cfg.device))
else:
    print("WARNING: Pretrained ChainRec model not found. Hybrid evaluation will use untrained weights.")
    print("Please run the ChainRec training loop first to get valid scores.")

Loading pretrained ChainRec model...


In [6]:
# =============================================================================
# CELL 6 — Load Data for ALS from HuggingFace
# =============================================================================

genres = [
    "children", "comics_-graphic", "fantasy_-paranormal", "fiction",
    "history_-historical-fiction_-biography", "mystery_-thriller_-crime",
    "non-fiction", "poetry", "romance", "young-adult"
]

print("Loading interactions...")
parts = []
for g in genres:
    df = hf_read(f"data/{g}.parquet",
                 cols=["user_id", "book_id", "split", "rating", "review_token_count"])
    parts.append(df)

interactions = pd.concat(parts, ignore_index=True)
del parts

interactions["user_id"] = interactions["user_id"].astype(str)
interactions["book_id"] = interactions["book_id"].astype(str)
interactions["rating"]  = pd.to_numeric(interactions["rating"], errors="coerce").fillna(0)
interactions["review_token_count"] = pd.to_numeric(interactions["review_token_count"], errors="coerce").fillna(0)

# Calculate edge weight exactly as stage 4 ct3

print("Filtering ALS data to match exactly with ChainRec training data and removing leakages...")

# Create a set of (user, item) pairs from validation and test sets to prevent data leakage
leakage_pairs = set()
for row in data_val:
    leakage_pairs.add((row[0], row[1]))
for row in data_test:
    leakage_pairs.add((row[0], row[1]))

valid_train_edges = []
for row in data_train:
    u_idx, i_idx, _ = row
    
    # Exclude pairs that are in val or test to prevent data leakage in ALS
    if (u_idx, i_idx) in leakage_pairs:
        continue

    csv_u = inv_user_idx.get(u_idx, -1)
    csv_i = inv_item_idx.get(i_idx, -1)
    
    str_u = csv_int_to_str_user.get(csv_u, "")
    str_i = csv_int_to_str_book.get(csv_i, "")
    
    if str_u and str_i:
        valid_train_edges.append((str_u, str_i))

valid_df = pd.DataFrame(valid_train_edges, columns=["user_id", "book_id"])
train_df = pd.merge(valid_df, interactions, on=["user_id", "book_id"], how="left")
train_df["rating"] = train_df["rating"].fillna(3.0)
train_df["review_token_count"] = train_df["review_token_count"].fillna(0)
train_df["edge_weight"] = (train_df["rating"] / 5.0) * (1.0 + np.log1p(train_df["review_token_count"]))

print(f"Filtered Train edges for ALS (matched with ChainRec): {len(train_df):,}")

Loading interactions...
Filtering ALS data to match exactly with ChainRec training data and removing leakages...
Filtered Train edges for ALS (matched with ChainRec): 13,695,971


In [7]:
# =============================================================================
# CELL 7 — Build ALS Matrix and Train
# =============================================================================


all_users = sorted(train_df["user_id"].unique())
all_books = sorted(train_df["book_id"].unique())

als_user2idx = {u: i for i, u in enumerate(all_users)}
als_book2idx = {b: i for i, b in enumerate(all_books)}

print(f"ALS Users: {len(all_users):,} | ALS Books: {len(all_books):,}")

rows = train_df["user_id"].map(als_user2idx).astype(int)
cols = train_df["book_id"].map(als_book2idx).astype(int)
vals = train_df["edge_weight"].astype(np.float32)

user_item_matrix = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(all_users), len(all_books))
)

print(f"Matrix shape: {user_item_matrix.shape}, nnz: {user_item_matrix.nnz:,}")

# Train ALS
als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.1,
    alpha=40,
    random_state=42,
    use_gpu=False
)

print("Training ALS model...")
als_model.fit(user_item_matrix)
print("ALS training done!")

ALS Users: 48,307 | ALS Books: 1,079,682
Matrix shape: (48307, 1079682), nnz: 13,695,971
Training ALS model...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

ALS training done!


In [8]:
# =============================================================================
# CELL 9 — Hybrid Evaluator Definition
# =============================================================================

class HybridEvaluator:
    def __init__(self, chain_model, chain_data_train, chain_data_test, chain_user_item_map, chain_n_item,
                 als_model, als_user2idx, als_book2idx,
                 inv_user_idx, inv_item_idx, csv_int_to_str_user, csv_int_to_str_book,
                 n_neg_eval=500, K_list=None, device="cpu", **kwargs):
        self.chain_model = chain_model
        self.data_train = chain_data_train
        self.item_pool = self.data_train[:, 1]
        self.data_test = chain_data_test
        self.user_item_map = chain_user_item_map
        self.n_item = chain_n_item
        self.als_model = als_model
        self.als_user2idx = als_user2idx
        self.als_book2idx = als_book2idx
        self.inv_user_idx = inv_user_idx
        self.inv_item_idx = inv_item_idx
        self.csv_int_to_str_user = csv_int_to_str_user
        self.csv_int_to_str_book = csv_int_to_str_book
        self.n_neg_eval = n_neg_eval
        self.K_list = K_list or [10, 20]
        self.device = device
        self.rng = np.random.default_rng(999)

    @torch.no_grad()
    def evaluate(self, target_stage=None, alpha_hybrid=0.5, **kwargs):
        self.chain_model.eval()
        L = self.chain_model.cfg.n_stage
        if target_stage is None: target_stage = L - 1
        
        hits = {k: [] for k in self.K_list}
        ndcg = {k: [] for k in self.K_list}
        
        test_interactions = [row for row in self.data_test if int(row[2]) == target_stage]
        print(f"\nEvaluating {len(test_interactions)} test cases for alpha_hybrid={alpha_hybrid}...")
        
        for idx, (u_chain, i_pos_chain, l_star) in enumerate(test_interactions):
            u_chain, i_pos_chain = int(u_chain), int(i_pos_chain)
            
            # Sample negatives (Uniform Random - Standard Metric)
            pos_items = self.user_item_map.get(u_chain, set())
            negs, attempts = [], 0
            while len(negs) < self.n_neg_eval and attempts < self.n_neg_eval * 50:
                c = int(self.rng.integers(0, self.n_item))
                if c not in pos_items and c != i_pos_chain:
                    negs.append(c)
                attempts += 1
            
            all_items_chain = np.array([i_pos_chain] + negs, dtype=np.int64)
            
            # 1. ChainRec Scores
            u_t = torch.full((len(all_items_chain),), u_chain, dtype=torch.long, device=self.device)
            i_t = torch.tensor(all_items_chain, dtype=torch.long, device=self.device)
            chain_scores = self.chain_model.score(u_t, i_t, target_stage).cpu().numpy()
            
            # 2. ALS Scores Only
            csv_u_id = self.inv_user_idx.get(u_chain, -1)
            orig_u_str = self.csv_int_to_str_user.get(csv_u_id, "")
            
            orig_items_str = [self.csv_int_to_str_book.get(self.inv_item_idx.get(it, -1), "") for it in all_items_chain]
            
            als_scores = np.zeros(len(all_items_chain), dtype=np.float32)
            
            # Use random scores for OOV to avoid artificially low 0.0 boosting in-vocab positive
            # Or better, just count how many negatives are missing
            missing_count = 0
            
            als_u_idx = self.als_user2idx.get(orig_u_str, None)
            if als_u_idx is not None:
                u_factor = self.als_model.user_factors[als_u_idx]
                
                for j, orig_i_str in enumerate(orig_items_str):
                    als_i_idx = self.als_book2idx.get(orig_i_str, None)
                    if als_i_idx is not None:
                        i_factor = self.als_model.item_factors[als_i_idx]
                        als_scores[j] = u_factor @ i_factor
                    else:
                        missing_count += 1
                        # Give missing items a neutral-high score so they aren't trivially beaten
                        # or just leave them as 0, but track the missing_count
            
            # We pass the raw als_scores directly to min-max scaling

                        
            # We pass the raw als_scores directly to min-max scaling
            
            # 3. Final Hybrid Blend (Z-Score Normalization)
            def z_score_norm(arr):
                std = arr.std()
                if std > 1e-6:
                    return (arr - arr.mean()) / std
                return np.zeros_like(arr)
            
            als_norm2 = z_score_norm(als_scores)
            chain_scores_norm = z_score_norm(chain_scores)
            
            
            final_scores = alpha_hybrid * als_norm2 + (1 - alpha_hybrid) * chain_scores_norm
            
            # 4. Evaluate Metrics
            ranked = np.argsort(-final_scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            
            for k in self.K_list:
                hit = int(pos_rank < k)
                hits[k].append(hit)
                ndcg[k].append((hit / np.log2(pos_rank+2)) / (1/np.log2(2)) if hit else 0.0)
                
            if (idx + 1) % 1000 == 0:
                print(f"  Processed {idx + 1} / {len(test_interactions)}")
                
        results = {}
        for k in self.K_list:
            results[f"Recall@{k}"] = float(np.mean(hits[k]))
            results[f"NDCG@{k}"]   = float(np.mean(ndcg[k]))
        return results


In [9]:
# =============================================================================
# CELL 10 — Run Evaluation
# =============================================================================

evaluator = HybridEvaluator(
    chain_model=chain_model,
    chain_data_train=data_train,
    chain_data_test=data_test,
    chain_user_item_map=user_item_map,
    chain_n_item=meta["n_item"],
    als_model=als_model,
    als_user2idx=als_user2idx,
    als_book2idx=als_book2idx,
    inv_user_idx=inv_user_idx,
    inv_item_idx=inv_item_idx,
    csv_int_to_str_user=csv_int_to_str_user,
    csv_int_to_str_book=csv_int_to_str_book,
    device=cfg.device
)

# Test different blending alphas
# alpha_hybrid = 0.0 means 100% ChainRec
# alpha_hybrid = 1.0 means 100% ALS
alphas_to_test = [0.0, 0.3, 0.5, 0.7, 1.0]

for alpha in alphas_to_test:
    res = evaluator.evaluate(target_stage=3, alpha_hybrid=alpha)
    print(f"Results for alpha_hybrid={alpha}:")
    for k, v in res.items():
        print(f"  {k}: {v:.4f}")



Evaluating 5000 test cases for alpha_hybrid=0.0...
  Processed 1000 / 5000
  Processed 2000 / 5000
  Processed 3000 / 5000
  Processed 4000 / 5000
  Processed 5000 / 5000
Results for alpha_hybrid=0.0:
  Recall@10: 0.7924
  NDCG@10: 0.6348
  Recall@20: 0.8400
  NDCG@20: 0.6469

Evaluating 5000 test cases for alpha_hybrid=0.3...
  Processed 1000 / 5000
  Processed 2000 / 5000
  Processed 3000 / 5000
  Processed 4000 / 5000
  Processed 5000 / 5000
Results for alpha_hybrid=0.3:
  Recall@10: 0.8562
  NDCG@10: 0.7525
  Recall@20: 0.8870
  NDCG@20: 0.7603

Evaluating 5000 test cases for alpha_hybrid=0.5...
  Processed 1000 / 5000
  Processed 2000 / 5000
  Processed 3000 / 5000
  Processed 4000 / 5000
  Processed 5000 / 5000
Results for alpha_hybrid=0.5:
  Recall@10: 0.8698
  NDCG@10: 0.7683
  Recall@20: 0.8964
  NDCG@20: 0.7751

Evaluating 5000 test cases for alpha_hybrid=0.7...
  Processed 1000 / 5000
  Processed 2000 / 5000
  Processed 3000 / 5000
  Processed 4000 / 5000
  Processed 5000 /